In [1]:
# import numpy as np
# import cupy as cp
import time
import random
import copy
from enum import Enum, auto
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

In [2]:
class GateType(Enum):
    X = auto()
    Y = auto()
    Z = auto()
    H = auto()

GATE_UNITARIES: Dict[GateType, List[List[complex]]]= {
    GateType.X: [[0, 1],
                [1, 0]],
    GateType.Y: [[0, -1j],
                [1j, 0]],
    GateType.Z: [[1, 0],
                [0, -1]],
    GateType.H: [[1/2**0.5, 1/2**0.5],
                [1/2**0.5, -1/2**0.5]],
}

NUM_QUBITS = 5
NUM_OPERATIONS = 20
INIT_MATRIX_SIZE = 2 * NUM_QUBITS + 1

@dataclass
class SingleQubitGate:
    gate: GateType
    a: complex
    b: complex
    c: complex
    d: complex
    target: int

@dataclass
class ControlledGate:
    gate: GateType
    a: complex
    b: complex
    c: complex
    d: complex
    target: int
    control: List[int]

GateOperation = SingleQubitGate | ControlledGate

def customize(gate: GateType, target: int, control: List[int] | None = None) -> GateOperation:
    matrix = GATE_UNITARIES[gate]
    a, b = matrix[0]
    c, d = matrix[1]
    if control == None:
        return SingleQubitGate(gate, a, b, c, d, target)
    else:
        return ControlledGate(gate, a, b, c, d, target, control)


def randomizer(option: Optional[GateOperation], threshold=0.5) -> List[GateOperation]:
    operations: List[GateOperation] = []

    for _ in range(NUM_OPERATIONS):
        gate = random.choice(list(GateType))
        matrix = GATE_UNITARIES[gate]
        a, b = matrix[0]
        c, d = matrix[1]
        
        if option == SingleQubitGate or random.random() < threshold:
            target = random.randint(0, NUM_QUBITS - 1)
            operations.append(SingleQubitGate(gate, a, b, c, d, target))
        elif option == ControlledGate or random.random() >= threshold:
            control, target = random.sample(range(NUM_QUBITS), 2)
            operations.append(ControlledGate(gate, a, b, c, d, target, [control]))
    
    return operations

def initialization(L, R) -> Tuple[List[List[complex]], List[List[complex]], List[List[int]]]:    
    if 1 < INIT_MATRIX_SIZE:
        L[1, 0] = 1  

    for col in range(1, INIT_MATRIX_SIZE - 2):  
        row = col + 2
        if row < INIT_MATRIX_SIZE:
            L[row, col] = 1

    if 2 < INIT_MATRIX_SIZE:
        R[2, 0] = 1  

    for col in range(1, INIT_MATRIX_SIZE - 2):  
        if col % 2 == 1:  
            row = col + 3
        else:  
            row = col + 2
        if row < INIT_MATRIX_SIZE:
            R[row, col] = 1

    Mapping = []
    Mapping.append([0])

    for i in range(1, NUM_QUBITS):
        Mapping.append([2*i - 1, 2*i])
    Mapping.append([2*i + 1])
    Mapping.append([2*i + 2])

    return L, R, Mapping


def _view_operations(operations: List[GateOperation]):
    for op in operations:
        print(op.gate.name, op.target)

In [3]:
operations = randomizer(option=ControlledGate)

# operations : List[GateOperation] = []
# operations.append(customize(GateType.X, 0))
# operations.append(customize(GateType.H, 0))
# operations.append(customize(GateType.X, 1, [0]))
# operations.append(customize(GateType.X, 2, [1]))

# operations.append(customize(GateType.Z, 2))

# _view_operations(operations)

In [4]:
import cupy as cp

def compute_result(L: cp.ndarray, R: cp.ndarray, seq: List[int], leaf: List[int]) -> complex:
    A = cp.zeros((1, L.shape[0]))
    for i in leaf:
        A[0, i] = 1

    B = cp.zeros((L.shape[0], 1))
    B[0, 0] = 1

    dirs = B
    for dir in seq:
        if dir == 0:
            dirs = L @ dirs
        else:
            dirs = R @ dirs
        
    result = A @ dirs

    return result.item()

In [5]:
cp_start = time.time()

In [6]:
L = cp.zeros((INIT_MATRIX_SIZE, INIT_MATRIX_SIZE), dtype=complex)
R = cp.zeros((INIT_MATRIX_SIZE, INIT_MATRIX_SIZE), dtype=complex)

L, R, Mapping = initialization(L, R)
# print(L)
# print(R)
# print(Mapping)

In [7]:
for op in operations:
    MATRIX_SIZE = L.shape[0]
    L_U, R_U = copy.deepcopy(L), copy.deepcopy(R)
    
    A = cp.eye(MATRIX_SIZE, dtype=complex)
    B = cp.zeros((MATRIX_SIZE, MATRIX_SIZE), dtype=complex)
    C = cp.zeros((MATRIX_SIZE, MATRIX_SIZE), dtype=complex)
    D = cp.eye(MATRIX_SIZE, dtype=complex)
    
    for node in Mapping[op.target]:
        A[node][node] = op.a
        B[node][node] = op.b
        C[node][node] = op.c
        D[node][node] = op.d

    L_U = L @ A + R @ B
    R_U = L @ C + R @ D

    if isinstance(op, SingleQubitGate):
        L, R = L_U, R_U
    
    elif isinstance(op, ControlledGate):
        from cupyx.scipy.linalg import block_diag
        L = block_diag(L, L_U)
        R = block_diag(R, R_U)
        
        if len(op.control) == 1:
            control = op.control[0]
            print(Mapping[control])
            for node in Mapping[control]:
                R[:, node] = R[:, node + MATRIX_SIZE]
                
            for mapping in Mapping:
                mapping_U = []
                for node in mapping:
                    mapping_U.append(node + MATRIX_SIZE)
                mapping += mapping_U
                
        else:
            for control in op.control:
                for node in Mapping[control]:
                    ...
                    
    if not L.shape == R.shape:
        raise ValueError("Size of L and R not equal")
    
    MATRIX_SIZE = L.shape[0]

[0]
[5, 6, 16, 17]
[0, 11, 22, 33]
[3, 4, 14, 15, 25, 26, 36, 37, 47, 48, 58, 59, 69, 70, 80, 81]
[1, 2, 12, 13, 23, 24, 34, 35, 45, 46, 56, 57, 67, 68, 78, 79, 89, 90, 100, 101, 111, 112, 122, 123, 133, 134, 144, 145, 155, 156, 166, 167]
[1, 2, 12, 13, 23, 24, 34, 35, 45, 46, 56, 57, 67, 68, 78, 79, 89, 90, 100, 101, 111, 112, 122, 123, 133, 134, 144, 145, 155, 156, 166, 167, 177, 178, 188, 189, 199, 200, 210, 211, 221, 222, 232, 233, 243, 244, 254, 255, 265, 266, 276, 277, 287, 288, 298, 299, 309, 310, 320, 321, 331, 332, 342, 343]
[1, 2, 12, 13, 23, 24, 34, 35, 45, 46, 56, 57, 67, 68, 78, 79, 89, 90, 100, 101, 111, 112, 122, 123, 133, 134, 144, 145, 155, 156, 166, 167, 177, 178, 188, 189, 199, 200, 210, 211, 221, 222, 232, 233, 243, 244, 254, 255, 265, 266, 276, 277, 287, 288, 298, 299, 309, 310, 320, 321, 331, 332, 342, 343, 353, 354, 364, 365, 375, 376, 386, 387, 397, 398, 408, 409, 419, 420, 430, 431, 441, 442, 452, 453, 463, 464, 474, 475, 485, 486, 496, 497, 507, 508, 518, 519,

KeyboardInterrupt: 

In [ ]:
cp_end = time.time()


In [ ]:
compute_result(L, R, [0] * NUM_QUBITS, Mapping[-2])

0j

In [ ]:
import numpy as np

def compute_result(L: np.ndarray, R: np.ndarray, seq: List[int], leaf: List[int]) -> complex:
    A = np.zeros((1, L.shape[0]))
    for i in leaf:
        A[0, i] = 1

    B = np.zeros((L.shape[0], 1))
    B[0, 0] = 1

    dirs = B
    for dir in seq:
        if dir == 0:
            dirs = L @ dirs
        else:
            dirs = R @ dirs
        
    result = A @ dirs

    return result.item()

In [ ]:
np_start = time.time()

In [ ]:
L = np.zeros((INIT_MATRIX_SIZE, INIT_MATRIX_SIZE), dtype=complex)
R = np.zeros((INIT_MATRIX_SIZE, INIT_MATRIX_SIZE), dtype=complex)

L, R, Mapping = initialization(L, R)
# print(L)
# print(R)
# print(Mapping)


In [ ]:
for op in operations:
    MATRIX_SIZE = L.shape[0]

    L_U, R_U = copy.deepcopy(L), copy.deepcopy(R)
    
    A = np.eye(MATRIX_SIZE, dtype=complex)
    B = np.zeros((MATRIX_SIZE, MATRIX_SIZE), dtype=complex)
    C = np.zeros((MATRIX_SIZE, MATRIX_SIZE), dtype=complex)
    D = np.eye(MATRIX_SIZE, dtype=complex)
    
    for node in Mapping[op.target]:
        A[node][node] = op.a
        B[node][node] = op.b
        C[node][node] = op.c
        D[node][node] = op.d

    L_U = L @ A + R @ B
    R_U = L @ C + R @ D

    if isinstance(op, SingleQubitGate):
        L, R = L_U, R_U
        
    elif isinstance(op, ControlledGate):
        from scipy.linalg import block_diag
        L = block_diag(L, L_U)
        R = block_diag(R, R_U)
        
        if len(op.control) == 1:
            control = op.control[0]
            print(Mapping[control])
            for node in Mapping[control]:
                R[:, node] = R[:, node + MATRIX_SIZE]
                
            for mapping in Mapping:
                mapping_U = []
                for node in mapping:
                    mapping_U.append(node + MATRIX_SIZE)
                mapping += mapping_U
                
        else:
            for control in op.control:
                for node in Mapping[control]:
                    ...

    if not L.shape == R.shape:
        raise ValueError("Size of L and R not equal")
    
    MATRIX_SIZE = L.shape[0]


[0]
[5, 6, 16, 17]
[7, 8, 18, 19, 29, 30, 40, 41]
[0, 11, 22, 33, 44, 55, 66, 77]
[5, 6, 16, 17, 27, 28, 38, 39, 49, 50, 60, 61, 71, 72, 82, 83, 93, 94, 104, 105, 115, 116, 126, 127, 137, 138, 148, 149, 159, 160, 170, 171]


In [ ]:
np_end = time.time()


In [ ]:
compute_result(L, R, [0] * NUM_QUBITS, Mapping[-2])


0j

In [ ]:
print(cp_end - cp_start)
print(np_end - np_start)

0.30739879608154297
0.05382990837097168


In [ ]:
print(L.shape)

(352, 352)
